In [1]:
import json
import pandas as pd

df = pd.read_csv(
    "all_proteomes_merged.tsv",
    sep="\t",          # TSV 的关键
    header=0,          # 第一行是列名（默认）
    encoding="utf-8",  # 常见编码：utf-8 / utf-8-sig / gbk
)

df.head()  # 查看前几行数据

,primaryAccession,uniProtkbId,organism,shortOrganism,entryType,proteinName,nameType,evidenceCode,evidenceSource,evidenceId,comment
0,A5A616,MGTS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Small protein MgtS,recommended,ECO:0000305,NaN,NaN,FUNCTION : Modulates intracellular Mg(2+) leve...
1,O32583,THIS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Sulfur carrier protein ThiS,recommended,NaN,NaN,NaN,FUNCTION : Is the sulfur donor in the synthesi...
2,O32583,THIS_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Thiamine biosynthesis protein ThiS,alternative,NaN,NaN,NaN,FUNCTION : Is the sulfur donor in the synthesi...
3,P00350,6PGD_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),"6-phosphogluconate dehydrogenase, decarboxylating",recommended,NaN,NaN,NaN,FUNCTION : Catalyzes the oxidative decarboxyla...
4,P00363,FRDA_ECOLI,Escherichia coli (strain K12),Escherichia_coli,UniProtKB reviewed (Swiss-Prot),Fumarate reductase flavoprotein subunit,recommended,NaN,NaN,NaN,"FUNCTION : Two distinct, membrane-bound, FAD-c..."


In [2]:
# 查看列名和数据
print("列名:", df.columns.tolist())
print("\n查看前几行的 proteinName:")
print(df[['proteinName']].head(10))

#查看有多少sample
print(df)

列名: ['primaryAccession', 'uniProtkbId', 'organism', 'shortOrganism', 'entryType', 'proteinName', 'nameType', 'evidenceCode', 'evidenceSource', 'evidenceId', 'comment']

查看前几行的 proteinName:
                                         proteinName
0                                 Small protein MgtS
1                        Sulfur carrier protein ThiS
2                 Thiamine biosynthesis protein ThiS
3  6-phosphogluconate dehydrogenase, decarboxylating
4            Fumarate reductase flavoprotein subunit
5     Quinol-fumarate reductase flavoprotein subunit
6              NADP-specific glutamate dehydrogenase
7                Type II NADH:quinone oxidoreductase
8                                   Cupric reductase
9                               NADH dehydrogenase-2
       primaryAccession  uniProtkbId                       organism  \
0                A5A616   MGTS_ECOLI  Escherichia coli (strain K12)   
1                O32583   THIS_ECOLI  Escherichia coli (strain K12)   
2              

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import json
from datasets import Dataset, DatasetDict
import numpy as np

# ============= 数据清洗和预处理 =============
print("="*80)
print("开始数据清洗和预处理")
print("="*80)

# 1. 删除缺失值
df = df.dropna(subset=["uniProtkbId", "proteinName", "nameType"])
print(f"删除缺失值后: {len(df)} 条记录")

# 2. 只保留 recommended 类型
df = df[df["nameType"].str.lower() == "recommended"].copy()
print(f"只保留 recommended 后: {len(df)} 条记录")

# 3. 高效检测 comment 泄露 product_name（使用向量化操作，比 apply 快 10x+）
def vectorized_leak_check(df):
    """向量化泄露检测，比逐行 apply 快得多"""
    protein_names = df['proteinName'].fillna('').str.lower()
    comments = df['comment'].fillna('').str.lower()
    # 使用 numpy 向量化操作
    leak_mask = np.array([
        pn in c if pn and c else False 
        for pn, c in zip(protein_names, comments)
    ])
    return leak_mask

leak_mask = vectorized_leak_check(df)
n_leaked = leak_mask.sum()
if n_leaked > 0:
    print(f"剔除 {n_leaked} 个 comment 泄露 product_name 的样本")
    df = df[~leak_mask].copy()
    
# 4. 创建规范化的列名
df["NAME"] = df["uniProtkbId"].astype(str).str.strip()
df["PRODUCT_NAME"] = df["proteinName"].astype(str).str.strip()

# 5. 去重
original_len = len(df)
df = df.drop_duplicates(subset=["NAME", "PRODUCT_NAME"])
print(f"去重后: {len(df)} 条记录 (删除 {original_len - len(df)} 条重复)")

# ============= 按 organism 分割数据集 =============
print("\n" + "="*80)
print("按 organism 分割数据集（确保不同集合的 organism 完全不重叠）")
print("="*80)

group_col = "shortOrganism" if "shortOrganism" in df.columns else "organism"
groups = df[group_col].astype(str)

# 使用 GroupShuffleSplit 确保同一 organism 的所有样本在同一集合
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=groups))
train = df.iloc[train_idx].copy()
temp = df.iloc[temp_idx].copy()

# dev/test 再按 group 切
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
groups_temp = temp[group_col].astype(str)
dev_idx, test_idx = next(gss2.split(temp, groups=groups_temp))
dev = temp.iloc[dev_idx].copy()
test = temp.iloc[test_idx].copy()

# 验证 organism 不重叠
train_organisms = set(train[group_col].unique())
dev_organisms = set(dev[group_col].unique())
test_organisms = set(test[group_col].unique())

print(f"\n训练集 organisms 数量: {len(train_organisms)}, 样本数: {len(train)}")
print(f"验证集 organisms 数量: {len(dev_organisms)}, 样本数: {len(dev)}")
print(f"测试集 organisms 数量: {len(test_organisms)}, 样本数: {len(test)}")

# 检查重叠
overlaps = [
    ("训练集 ∩ 验证集", train_organisms & dev_organisms),
    ("训练集 ∩ 测试集", train_organisms & test_organisms),
    ("验证集 ∩ 测试集", dev_organisms & test_organisms),
]
print(f"\n验证 organism 不重叠:")
all_clean = True
for name, overlap in overlaps:
    print(f"  {name}: {len(overlap)} 个 organisms")
    if overlap:
        all_clean = False
        
if all_clean:
    print("\n✓ 确认: 训练集、验证集、测试集的 organisms 完全不重叠!")
else:
    print("\n⚠️ 警告: 发现 organism 重叠!")

# ============= 生成 3 个阶梯级别的数据集 =============
print("\n" + "="*80)
print("生成 3 个阶梯级别的数据集")
print("="*80)

# 创建采样子集（用于 dev/test）
dev_subset = dev.sample(n=min(10000, len(dev)), random_state=42).reset_index(drop=True)
test_subset = test.sample(n=min(10000, len(test)), random_state=42).reset_index(drop=True)

def create_and_save_dataset(train_df, dev_df, test_df, columns, save_path, level_name):
    """统一的数据集创建和保存函数"""
    # 处理缺失值（主要是 comment 字段）
    for col in columns:
        if col == "comment":
            train_df = train_df.copy()
            dev_df = dev_df.copy()
            test_df = test_df.copy()
            train_df["comment"] = train_df["comment"].fillna("")
            dev_df["comment"] = dev_df["comment"].fillna("")
            test_df["comment"] = test_df["comment"].fillna("")
    
    train_ds = Dataset.from_pandas(train_df[columns], preserve_index=False)
    dev_ds = Dataset.from_pandas(dev_df[columns], preserve_index=False)
    test_ds = Dataset.from_pandas(test_df[columns], preserve_index=False)
    
    dataset_dict = DatasetDict({
        "train": train_ds,
        "dev": dev_ds,
        "test": test_ds
    })
    dataset_dict.save_to_disk(save_path)
    print(f"[{level_name}] Train: {len(train_ds)}, Dev: {len(dev_ds)}, Test: {len(test_ds)}")
    print(f"  已保存到: {save_path}/")
    return dataset_dict

# Level 1: NAME, PRODUCT_NAME
print("\n[Dataset 1] 字段: NAME, PRODUCT_NAME")
dataset_dict_1 = create_and_save_dataset(
    train, dev, test, 
    ["NAME", "PRODUCT_NAME"], 
    "protein_dataset_level1_full", "Level 1"
)

# Level 2: NAME, organism, PRODUCT_NAME
print("\n[Dataset 2] 字段: NAME, organism, PRODUCT_NAME")
dataset_dict_2 = create_and_save_dataset(
    train, dev, test,
    ["NAME", "organism", "PRODUCT_NAME"],
    "protein_dataset_level2_full", "Level 2"
)

# Level 3: NAME, organism, comment, PRODUCT_NAME (使用采样的 dev/test)
print("\n[Dataset 3] 字段: NAME, organism, comment, PRODUCT_NAME")
dataset_dict_3 = create_and_save_dataset(
    train, dev_subset, test_subset,
    ["NAME", "organism", "comment", "PRODUCT_NAME"],
    "protein_dataset_level3_full", "Level 3"
)

print("\n" + "="*80)
print("✓ 所有3个数据集生成完成！")
print("="*80)

In [26]:
# 加载并验证三个数据集
from datasets import load_from_disk

print("\n" + "="*80)
print("加载并验证三个数据集")
print("="*80)



# 加载 Dataset 2
loaded_ds2 = load_from_disk("/users/thz501/fastscratch/bio/dataset/protein_dataset_level3_full")
print("\n[Dataset 2] 字段: NAME, organism, PRODUCT_NAME")
print(f"  总体结构: {loaded_ds2}")
print(f"  Train 示例:")
print(f"    {loaded_ds2['train'][0]}")





加载并验证三个数据集

[Dataset 2] 字段: NAME, organism, PRODUCT_NAME
  总体结构: DatasetDict({
    train: Dataset({
        features: ['NAME', 'organism', 'comment', 'PRODUCT_NAME'],
        num_rows: 297171
    })
    dev: Dataset({
        features: ['NAME', 'organism', 'comment', 'PRODUCT_NAME'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['NAME', 'organism', 'comment', 'PRODUCT_NAME'],
        num_rows: 10000
    })
})
  Train 示例:
    {'NAME': 'MGTS_ECOLI', 'organism': 'Escherichia coli (strain K12)', 'comment': 'FUNCTION : Modulates intracellular Mg(2+) levels to maintain cellular integrity upon Mg(2+) limitation. Acts by binding and stabilizing the Mg(2+) transporter MgtA, thereby leading to increased intracellular level of Mg(2+). May inhibit FtsH proteolysis of MgtA ; SUBUNIT : Interacts with MgtA ; SUBCELLULAR LOCATION : INDUCTION : Induced by very low Mg(2+) via the PhoQ/PhoP two-component regulatory system (PubMed:28512220). Induced by SDS/EDTA (envelope stress), 

In [7]:
# 对新的 dataset 做处理 (Level 3: Gene_ID + user_prompt -> Product_Description)
import json
import re
from datasets import Dataset, DatasetDict

def clean_user_prompt(user_prompt: str) -> str:
    """
    清理 user_prompt：
    - 移除 'Gene Summary:' 前缀
    - 移除最后一行 'Based on this summary, generate a product description for ...' 
    - 保留 Gene ID 信息
    """
    # 去掉首尾空白
    text = user_prompt.strip()
    # 移除 'Gene Summary:' 前缀
    text = re.sub(r'^Gene Summary:\s*', '', text, flags=re.IGNORECASE)
    
    # 移除 "Based on this summary..." 这一行
    text = re.sub(r'\nBased on this summary.*$', '', text, flags=re.IGNORECASE)
    
    return text.strip()


def clean_label(correct_answer: str) -> str:
    """
    清理 correct_answer：
    - 移除 'Product_Description: ' 前缀
    - 移除 ', putative Evidence_Code: ISS'
    - 移除 ' Evidence_Code: ISS'
    """
    label = correct_answer.strip()
    
    # 移除 'Product_Description: ' 前缀
    if label.startswith("Product_Description:"):
        label = label[len("Product_Description:"):].strip()
    
    # 移除 ', putative Evidence_Code: ISS'
    label = re.sub(r',?\s*putative\s+Evidence_Code:\s*ISS$', '', label, flags=re.IGNORECASE)
    
    # 移除 ' Evidence_Code: ISS'
    label = re.sub(r'\s*Evidence_Code:\s*ISS$', '', label, flags=re.IGNORECASE)
    
    return label.strip()


def process_dataset(json_path: str) -> list:
    """
    读取并处理 JSON 数据集
    返回处理后的样本列表
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    processed = []
    for sample in data:
        processed_sample = {
            "Gene_ID": sample["Gene_ID"],
            "PMID": sample["PMID"],
            "user_prompt": clean_user_prompt(sample["user_prompt"]),
            "PRODUCT_NAME": clean_label(sample["correct_answer"])
        }
        processed.append(processed_sample)
    
    return processed


# 测试清理函数
test_answer_1 = "Product_Description: Identified spore protein 6 Evidence_Code: ISS"
test_answer_2 = "Product_Description: periodic tryptophan protein 2, putative Evidence_Code: ISS"
test_answer_3 = "Product_Description: ATP dependent helicase, putative Evidence_Code: ISS"

print("测试 label 清理:")
print(f"  原始: {test_answer_1}")
print(f"  清理后: {clean_label(test_answer_1)}")
print()
print(f"  原始: {test_answer_2}")
print(f"  清理后: {clean_label(test_answer_2)}")
print()
print(f"  原始: {test_answer_3}")
print(f"  清理后: {clean_label(test_answer_3)}")

测试 label 清理:
  原始: Product_Description: Identified spore protein 6 Evidence_Code: ISS
  清理后: Identified spore protein 6

  原始: Product_Description: periodic tryptophan protein 2, putative Evidence_Code: ISS
  清理后: periodic tryptophan protein 2

  原始: Product_Description: ATP dependent helicase, putative Evidence_Code: ISS
  清理后: ATP dependent helicase


In [8]:
# 处理 train/validation/test 数据集
import os

# 数据集路径
train_path = "training_dataset.json"
valid_path = "validation_dataset.json"
test_path = "test_dataset.json"

print("="*80)
print("处理新的 Gene Summary 数据集 (Level 3)")
print("="*80)

# 处理各个数据集
train_data = process_dataset(train_path)
valid_data = process_dataset(valid_path)
test_data = process_dataset(test_path)

print(f"\nNumber of training samples: {len(train_data)}")
print(f"Number of validation samples: {len(valid_data)}")
print(f"Number of test samples: {len(test_data)}")

# 显示示例
print("\n" + "="*80)
print("The first sample in the training set:")
print("="*80)
sample = train_data[0]
print(f"Gene_ID: {sample['Gene_ID']}")
print(f"PMID: {sample['PMID']}")
print(f"PRODUCT_NAME (label): {sample['PRODUCT_NAME']}")
print(f"\nuser_prompt (first 500 characters):\n{sample['user_prompt'][:500]}...")

处理新的 Gene Summary 数据集 (Level 3)

Number of training samples: 3538
Number of validation samples: 2128
Number of test samples: 1887

The first sample in the training set:
Gene_ID: CNA04360
PMID: 26313153
PRODUCT_NAME (label): Identified spore protein 6

user_prompt (first 500 characters):
• ISP6 (CNA04360) is one of 18 spore-enriched proteins identified through proteomic analysis that were consistently detected in all three spore replicates but never in yeast samples.
• ISP6 is classified as a Cryptococcus-specific protein containing a transmembrane domain with no conserved functional domains identified.
• Deletion of ISP6 (isp6Δ) resulted in no detectable phenotype in any assays performed, including yeast growth, sexual development, spore formation, spore stress resistance, spor...


In [ ]:
# 创建 HuggingFace Dataset 并保存到磁盘
from datasets import Dataset, DatasetDict
import numpy as np
# 转换为 Dataset
train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)
test_ds = Dataset.from_list(test_data)


# train 和 valid merge 后按照9：1 划分
combined_data = train_data + valid_data
np.random.seed(42)
np.random.shuffle(combined_data)
split_idx = int(0.9 * len(combined_data))
train_data = combined_data[:split_idx]
valid_data = combined_data[split_idx:]

train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)

# 创建 DatasetDict
dataset_dict = DatasetDict({
    "train": train_ds,
    "dev": valid_ds,  # 使用 dev 以保持与之前数据集一致
    "test": test_ds
})

# 保存到磁盘
output_dir = "/users/thz501/fastscratch/bio/dataset/gene_summary_dataset"
dataset_dict.save_to_disk(output_dir)

print(f"\n数据集已保存到: {output_dir}/")
print(f"\n数据集结构:")
print(dataset_dict)
print(f"\n字段: {train_ds.column_names}")

Saving the dataset (1/1 shards): 100%|██████████| 1887/1887 [00:00<00:00, 316535.42 examples/s]


数据集已保存到: /users/thz501/fastscratch/bio/dataset/gene_summary_dataset_level3/

数据集结构:
DatasetDict({
    train: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 5099
    })
    dev: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 567
    })
    test: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 1887
    })
})

字段: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME']


In [25]:
# 验证保存的数据集
from datasets import load_from_disk

print("="*80)
print("加载并验证数据集")
print("="*80)

loaded_ds = load_from_disk(output_dir)
print(f"\n加载成功: {loaded_ds}")

# 显示各集合的示例
for split_name in ["train", "dev", "test"]:
    print(f"\n[{split_name}] 示例:")
    sample = loaded_ds[split_name][0]
    print(f"  Gene_ID: {sample['Gene_ID']}")
    print(f"  PMID: {sample['PMID']}")
    print(f"  PRODUCT_NAME: {sample['PRODUCT_NAME']}")
    print(f"  user_prompt: {sample['user_prompt']} characters")

    # 每个集合的user_prompt长度统计
    prompt_lengths = [len(s['user_prompt']) for s in loaded_ds[split_name]]
    print(f"  user_prompt 长度统计:")
    print(f"    最小值: {np.min(prompt_lengths)}")
    print(f"    最大值: {np.max(prompt_lengths)}")
    print(f"    平均值: {np.mean(prompt_lengths):.2f}")
    # 查看 75% 分位数
    print(f"    分位数: {np.percentile(prompt_lengths, 96)}")

加载并验证数据集

加载成功: DatasetDict({
    train: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 5099
    })
    dev: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 567
    })
    test: Dataset({
        features: ['Gene_ID', 'PMID', 'user_prompt', 'PRODUCT_NAME'],
        num_rows: 1887
    })
})

[train] 示例:
  Gene_ID: PRCDC_0217300
  PMID: 26525978
  PRODUCT_NAME: ATP-dependent RNA helicase DDX47
  user_prompt: • DDX47 (also called Rrp8p or PfRrp8p in P. falciparum) is predicted to function in 18S rRNA processing, specifically participating in cleavages at A2, and to a lesser extent, A0 and A1 sites during ribosome biogenesis.
• DDX47 is classified as an RNA helicase involved in ribosome biogenesis, specifically in small subunit pre-processing.
• The gene is identified as PF3D7_0218400 in P. falciparum and belongs to the DExD/H helicase family involved in RNA metabolism.

Gene ID: PRCDC_0217

In [14]:
from datasets import load_from_disk

ds2 = load_from_disk("/users/thz501/fastscratch/bio/dataset/gene_summary_uniprot_style")

ds2["test"][5]



{'Gene_ID': 'CND02990',
 'PMID': 18854164,
 'user_prompt': 'FUNCTION : Encodes a LAG1 family DNA-binding regulator involved in the regulation of melanization in Cryptococcus neoformans. Plays a role in melanin production ; SUBCELLULAR LOCATION : Nucleus ; SIMILARITY : Belongs to the LAG1 family.',
 'PRODUCT_NAME': 'LAG1 family transcription factor',
 'user_prompt_original': '• PAN1 (gene ID: CND02990, also known as 5207.AAW43213, AAW43213.1, CNAG_01173) encodes a LAG1 family DNA-binding regulator.\n• Deletion of PAN1 (pan1Δ) resulted in a melanization defect, with the mutant displaying reduced melanin production compared to wild-type.\n• The pan1Δ mutant had an STM score of 1.4, indicating slightly increased infectivity compared to wild-type, though this did not meet the threshold for significant increased infectivity (>2.5).\n• The pan1Δ mutant showed a growth score of -0.40 in minimal medium at 37°C, indicating a mild growth defect that did not reach the significance threshold of -0.

{'Gene_ID': 'PRCDC_0217300',
 'PMID': 26525978,
 'user_prompt': '• DDX47 (also called Rrp8p or PfRrp8p in P. falciparum) is predicted to function in 18S rRNA processing, specifically participating in cleavages at A2, and to a lesser extent, A0 and A1 sites during ribosome biogenesis.\n• DDX47 is classified as an RNA helicase involved in ribosome biogenesis, specifically in small subunit pre-processing.\n• The gene is identified as PF3D7_0218400 in P. falciparum and belongs to the DExD/H helicase family involved in RNA metabolism.\n\nGene ID: PRCDC_0217300',
 'PRODUCT_NAME': 'ATP-dependent RNA helicase DDX47'}